# Stacking

### Stacking

- Stacking is a proven technique for improving prediction accuracy of standalone models. 

### Training Base Estimators

**K-Fold Cross Validation**:
- Avoids overffiting of model estimator combinations

- Consider 10 segments (or folds) and a stacking model that uses a logistic regression model and a decision tree model. Each estimator can be trained using data from 9 of the segments, and make predictions on the excluded 10th segment.

- We then append the predictions as new features to that 10th segment.
*Now 1/10th of the training data has two new features: one is the prediction made by the logistic regression model and the other is the prediction made by the decision tree model.*

- do the same with the other 9 segments, so we rotate the excluded segment and repeat this process until all training data points are augmented with new features. The end result is a prediction made on each training sample, without having seen the sample during the training process.

![k-fold diagram](images/k-fold%20diagram.png)


**Feature Augemnetation**:
- In our stacking setup, the base estimators need to be trained to make predictions on our training data. The prediction of each estimator will be appended to the corresponding data sample as a new feature. We thus augment the training data set with this additional information. The augmented training set is used by our later-stage stacking model to make the final prediction.

![stacking diagram](images/stacking%20diagram.png)

**example**
*in summary, say our training dataset has 10,000 samples, 10 features, and we select 3 base estimators.* 

1. Train each base estimator on the training set and make predictions on the training set.

2. Each estimator would make a prediction on each sample. (each sample will have 3 predictions)

3. These 3 predictions are appended to the pre-existing 10 features, (10,000 training samples with 13 features each)
---

**Training Stacking Model**:

```python
# import libraries:
import pandas as pd

# import dataset:
df = pd.read_csv('water_potability')
print(df.columns, df.shape)

# split dataset:
X = water_potability.drop(['Potability'], axis=1)
y = water_potability['Potability']

# train-test split:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=rand_state)

# assemble the ensemble:
level_0_estimators = dict()
level_0_estimators["logreg"] = LogisticRegression(random_state=rand_state)
level_0_estimators["forest"] = RandomForestClassifier(random_state=rand_state)

level_0_columns = [f"{name}_prediction" for name in level_0_estimators.keys()]

level_1_estimator = RandomForestClassifier(random_state=rand_state)
```
---

**k-fold cross-validation**:
- `sklearn.model_selection.StratifiedKFold` from the `scikit-learn` library. The kfold is then given to the instantiated `StackingClassifier`

```python
# k-fold cross-validation
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=rand_state)
stacking_clf = StackingClassifier(estimators=list(level_0_estimators.items()), 
                                    final_estimator=level_1_estimator, 
                                    passthrough=True, cv=kfold, stack_method="predict_proba")
```

**`fit_transform`**
- handles training of base estimators
- makes cross-validation predictions on training set
- augment training set with predictions from each estimator

```python
df = pd.DataFrame(stacking_clf.fit_transform(X_train, y_train), columns=level_0_columns + list(X_train.columns))
```
---

**predictions**

```python
y_val_pred = stacking_clf.predict(X_test)
stacking_accuracy = accuracy_score(y_test, y_val_pred)

vanilla_logistic_regression = LogisticRegression(random_state=rand_state).fit(X_train, y_train)
lr_accuracy = accuracy_score(y_test, vanilla_logistic_regression.predict(X_test))
                                   
vanilla_decision_tree = RandomForestClassifier(random_state=rand_state).fit(X_train, y_train)
dt_accuracy =  accuracy_score(y_test, vanilla_decision_tree.predict(X_test))

print(f'Stacking accuracy: {stacking_accuracy:.4f}')
print(f'Logistic Regression accuracy: {lr_accuracy:.4f}')
print(f'Decision Tree accuracy: {dt_accuracy:.4f}')
```

>  Enhance model diversity by using different training algorithms, different training sets, different feature subsets, and different hyperparameters.

